# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises are recommended to alleviate discomfort and prevent future episodes of lower back pain.'

In [12]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that help regulate growth and appetite. Adequate sleep (7-9 hours per night) supports immune function, aids in recovery, and helps maintain a healthy weight. Poor sleep or sleep disorders like insomnia can negatively impact health, affecting mood, increasing the risk of chronic conditions, and reducing overall quality of life. Therefore, maintaining good sleep hygiene and a consistent sleep routine is essential for promoting overall health and wellness.'

In [13]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing immediate stress relief techniques like deep breathing, progressive muscle relaxation, or grounding exercises\n- Taking short walks, preferably in nature\n- Listening to calming music\n\nThese approaches can help manage stress and alleviate headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Some exercises that can help alleviate lower back pain include:\n\n- **Cat-Cow Stretch:** Begin on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From your hands and knees, extend opposite arm and leg, keeping your core engaged. Hold for about 5 seconds, then switch sides. Aim for 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help relieve lower back discomfort and prevent future episodes.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health. Maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and ensuring a comfortable sleep environment can improve sleep quality. Good sleep hygiene practices—such as keeping the bedroom cool, dark, and quiet, limiting screen time before bed, and avoiding caffeine and heavy meals late in the day—help promote restorative sleep. Quality sleep supports immune function, mental health, and proper nutrient absorption, and it can boost energy, mood, and productivity throughout the day. Conversely, poor sleep or insomnia can negatively impact many aspects of health, making good sleep practices essential for overall wellness.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as deep breathing exercises and progressive muscle relaxation. Herbal teas like chamomile or valerian root may also help reduce stress and promote relaxation. Additionally, practicing meditation and ensuring proper hydration can help manage stress-related headaches.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

Lets say user has query - "What does the document say about vitamin B12 deficiency symptoms?
BM25 is Better Here, BM25 is better when the exact words matter.

If the document literally contains the phrase “vitamin B12 deficiency symptoms”, BM25 will match those exact terms very strongly. But
Embeddings might:
- Paraphrase it
- Focus on semantic similarity
- Miss exact keyword-heavy matches

BM25 is better when the query is keyword specific and we care about exact term matching. It shines when users search using precise words that are directly present in the document.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back upward (cat pose) and letting it sag downward (cow pose). Repeat 10-15 times.\n- **Bird Dog:** From hands and knees, extend your opposite arm and leg simultaneously while engaging your core. Hold each extension for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abdominal muscles, and tilt your pelvis to flatten your lower back against the floor. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretches and strengthening exercises can help alleviate discomfort and prevent future episodes.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health in several ways. It is crucial for physical repair, as during sleep the body repairs tissues and regenerates cells. Sleep also supports mental well-being and cognitive function by helping consolidate memories and process information. Additionally, sleep regulates hormones that control growth and appetite. Adults generally need 7-9 hours of quality sleep per night, which occurs in cycles involving REM and non-REM stages, each playing a role in physical and mental restoration. Creating a comfortable sleep environment and managing sleep disorders like insomnia are important for maintaining good health.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in progressive muscle relaxation, using grounding techniques such as identifying objects around you, taking short walks preferably in nature, listening to calming music, staying hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gently massaging your temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [24]:
#from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [25]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Based on the provided information, exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch: Begin on your hands and knees. Alternate between arching your back up (cat position) and letting it sag down (cow position). Perform 10-15 repetitions.\n\n2. Bird Dog: From a hands-and-knees position, extend the opposite arm and leg simultaneously while engaging your core. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. Partial Crunches: Lie on your back with knees bent. Cross your arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Hold briefly, then lower back down. Do 8-12 repetitions.\n\n4. Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold the stretch for 15-30 seconds, then switch legs.\n\n5. Pelvic Tilts: Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis up slightly

In [27]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Proper sleep is crucial for physical health, mental well-being, and cognitive function. During sleep, your body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night for adults, supports immune function, reduces stress, and enhances recovery from illness. Conversely, poor sleep or sleep disturbances can lead to issues such as fatigue, stress-related symptoms, weakened immunity, and increased risk for health problems. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are important strategies for promoting overall health.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing deep breathing, progressive muscle relaxation, grounding techniques, taking short walks in nature, and listening to calming music. For headaches, effective natural remedies include drinking plenty of water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, massaging temples and neck muscles, and using essential oils like peppermint or lavender.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

When we generate multiple reformulations of the same user query, we are basically asking the same question in different ways.
Sometimes the documents don’t contain the exact wording the user used. So if we search using only one version of the question, we might miss relevant documents. By creating multiple variations of the query, we increase the chances that at least one version matches how the information is written in the documents.
So overall, this improves recall because we retrieve more relevant documents that might have been missed with just a single query.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [29]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, some recommended exercises include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Repeat 10-15 times.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg, hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten your stomach muscles, and lift your shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nRemember to perform these exercises gently and consult with a healthcare professional before starting any new exercise routine, especia

In [35]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that help regulate growth and appetite. Adequate sleep — typically 7-9 hours per night for adults — is essential for maintaining energy levels, supporting immune function, and facilitating emotional stability. Poor sleep quality or insufficient sleep can lead to fatigue, impaired concentration, increased stress levels, and a higher risk of health issues such as cardiovascular disease, weakened immunity, and mental health problems. Therefore, prioritizing good sleep hygiene and ensuring restorative sleep is fundamental to maintaining overall health.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises\n- Progressive muscle relaxation\n- Grounding techniques (noticing sensory details around you)\n- Taking short walks, especially in nature\n- Listening to calming music\n- Applying peppermint or lavender essential oils\n- Drinking herbal teas such as chamomile or valerian root\n- Ensuring adequate sleep and maintaining a regular sleep schedule\n- Staying hydrated and drinking plenty of water\n- Gentle massage of the temples and neck\n- Using warm or cold compresses on the head or neck\n\nThese approaches can help alleviate headaches and manage stress naturally.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
#from langchain.retrievers import EnsembleRetriever
from langchain_classic.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nRemember to perform these exercises gently and consult a healthcare professional if you have ongoin

In [40]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive well-being. During sleep, your body repairs tissues, regenerates cells, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours for adults—helps improve immune function, enhances memory and learning, and maintains emotional stability. Poor or insufficient sleep can lead to health issues such as fatigue, stress, weakened immunity, and increased risk of chronic conditions. Therefore, maintaining good sleep hygiene and creating a conducive sleep environment are essential for overall health and wellness.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., inhale for 4 counts, hold for 4, exhale for 4)\n- Progressive muscle relaxation (tensing and relaxing muscle groups)\n- Grounding techniques (naming objects you see, hear, feel, smell, and taste)\n- Taking short walks, preferably in nature\n- Listening to calming music\n- For headaches specifically:\n  - Drink water and stay hydrated\n  - Apply cold or warm compresses to the head or neck\n  - Rest in a dark, quiet room\n  - Gentle massage of temples and neck\n  - Use peppermint or lavender essential oils\n  - Maintain a regular sleep schedule\n  - Limit caffeine intake in small amounts (be aware it can help or hurt)\n  \nAdditionally, practicing mindfulness and relaxation techniques can help reduce stress overall.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [44]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross your arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, tighten your abs and tilt your pelvis up slightly to flatten your back against the floor. Hold for 10 seconds and repeat 8-12 times.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides.\n\nIncorporating these gentle stretching and strengthening exercises can alleviate discomfort and help prevent future episodes of lower ba

In [48]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adults generally need 7-9 hours of sleep per night, with sleep occurring in cycles that include REM and non-REM stages. Quality sleep is influenced by good sleep hygiene practices, such as maintaining a consistent schedule, creating a relaxing bedtime routine, and optimizing the sleep environment (cool, dark, quiet). Poor sleep or sleep disturbances like insomnia can negatively impact physical health, cognitive performance, mood, immune function, and increase the risk for chronic conditions. Therefore, adequate and restorative sleep is fundamental for maintaining overall health and well-being.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include deep breathing exercises, progressive muscle relaxation, grounding techniques (such as identifying things you see, hear, feel, smell, and taste), taking short walks, listening to calming music, practicing mindfulness and meditation, and engaging in hobbies or leisure activities. \n\nFor headaches, natural remedies involve staying well-hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark and quiet environment, gentle massage of temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

If sentences are short and highly repetitive, like in FAQs, semantic chunking may not work very effectively. Because many sentences will have very similar embeddings, the similarity scores between them will also be very close. That means the algorithm may not find strong “breakpoints” and could either:
 - Merge too many sentences into one big chunk
 - Or create chunks that don’t meaningfully separate topics
To adjust the algorithm, we could:
 - Lower the breakpoint threshold so it splits more aggressively
 - Or combine semantic chunking with a fixed chunk size limit
 - Or increase sensitivity (like changing percentile settings)

Basically, when content is repetitive, we need to tune the threshold so the chunker doesn’t treat everything as one big similar block.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

To perform Activity #1, I first created the golden dataset, I created a grounded and document-aware manual sample instead of using **automated synthetic generation**. This decision was made to avoid version compatibility issues (I was facing), ensure reproducibility, and control API costs. Since the goal of this activity was to compare retriever performance rather than evaluate data generation quality, this approach provided a stable and cost-efficient evaluation setup.

In [50]:
golden_dataset = [
    {
        "question": "What are the benefits of regular physical activity?",
        "ground_truth": "Regular physical activity improves brain health, helps manage weight, reduces disease risk, strengthens bones and muscles, and improves daily functioning."
    },
    {
        "question": "What are the four main types of exercise?",
        "ground_truth": "The four main types of exercise are aerobic (cardio), strength training, flexibility, and balance exercises."
    },
    {
        "question": "How much aerobic exercise should adults aim for each week?",
        "ground_truth": "Adults should aim for at least 150 minutes of moderate-intensity aerobic activity per week."
    },
    {
        "question": "How often should adults do muscle-strengthening activities?",
        "ground_truth": "Adults should perform muscle-strengthening activities on two or more days per week."
    },
    {
        "question": "How common is lower back pain among adults?",
        "ground_truth": "Lower back pain affects approximately 80% of adults at some point in their lives."
    },
    {
        "question": "How can lower back pain be alleviated?",
        "ground_truth": "Gentle stretching and strengthening exercises can help relieve lower back pain and prevent future episodes."
    },
    {
        "question": "Why is exercise important for overall health?",
        "ground_truth": "Exercise supports overall health by improving physical and mental well-being and reducing disease risk."
    },
    {
        "question": "What should a well-rounded fitness routine include?",
        "ground_truth": "A well-rounded fitness routine includes aerobic, strength training, flexibility, and balance exercises."
    },
    {
        "question": "How does exercise help with weight management?",
        "ground_truth": "Exercise helps manage weight by increasing physical activity and energy expenditure."
    },
    {
        "question": "What are some benefits of strengthening bones and muscles?",
        "ground_truth": "Strengthening bones and muscles improves physical function and reduces injury risk."
    }
]

In [51]:
from datasets import Dataset

questions = [item["question"] for item in golden_dataset]
ground_truths = [item["ground_truth"] for item in golden_dataset]

evaluation_data = Dataset.from_dict({
    "question": questions,
    "ground_truth": ground_truths,
})

In [52]:
from ragas import evaluate
from ragas.metrics import context_precision, context_recall

def evaluate_retriever(retriever, name):
    retrieved_contexts = []

    for q in questions:
        docs = retriever.invoke(q)
        retrieved_contexts.append([doc.page_content for doc in docs])

    dataset_with_context = Dataset.from_dict({
        "question": questions,
        "ground_truth": ground_truths,
        "contexts": retrieved_contexts
    })

    result = evaluate(
        dataset_with_context,
        metrics=[context_precision, context_recall],
    )

    print(f"Results for {name}")
    print(result)
    
    return result

bm25_results = evaluate_retriever(bm25_retriever, "BM25")

C:\Users\Manish Kumar\AppData\Local\Temp\ipykernel_33288\1438113217.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
C:\Users\Manish Kumar\AppData\Local\Temp\ipykernel_33288\1438113217.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Results for BM25
{'context_precision': 0.8333, 'context_recall': 0.9000}


In [53]:
naive_results = evaluate_retriever(naive_retriever, "Naive Vector")

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Results for Naive Vector
{'context_precision': 0.8671, 'context_recall': 1.0000}


In [54]:
parent_results = evaluate_retriever(parent_document_retriever, "Parent Retriever")

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Results for Parent Retriever
{'context_precision': 0.9833, 'context_recall': 1.0000}


In [55]:
compression_results = evaluate_retriever(compression_retriever, "Compression Retriever")

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Results for Compression Retriever
{'context_precision': 1.0000, 'context_recall': 1.0000}


In [56]:
multi_query_results = evaluate_retriever(multi_query_retriever, "MultiQuery Retriever")

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Results for MultiQuery Retriever
{'context_precision': 0.9165, 'context_recall': 1.0000}


In [58]:
ensemble_results = evaluate_retriever(ensemble_retriever, "Ensemble Retriever")

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Results for Ensemble Retriever
{'context_precision': 0.5740, 'context_recall': 1.0000}


## Activity #1 Summary

I evaluated six different retrievers using context precision and context recall as the primary metrics, while also observing runtime as a proxy for latency and cost.

- The ParentDocumentRetriever performed extremely well, achieving a precision of 0.9833 and recall of 1.0000 in just 2m 8s. This makes it a strong balance between performance and efficiency.

- The Compression Retriever achieved perfect precision and recall (1.0000 / 1.0000) in 2m 9s. However, since it uses an additional reranking step, it introduces extra cost despite similar runtime in this small experiment.

- The Naive Vector Retriever achieved full recall (1.0000) with moderate precision (0.8671), but took longer (5m 39s), likely due to larger retrieval size (k=10).

- The MultiQuery Retriever also achieved full recall with improved precision (0.9165), but took significantly longer (8m 44s) because it generates multiple reformulated queries per question, increasing LLM calls and cost.

- The BM25 Retriever was relatively fast (2m 39s) but had lower recall (0.9000), meaning it missed some relevant contexts compared to embedding-based methods.

- The Ensemble Retriever achieved full recall but had very low precision (0.5740) and the highest runtime (11m 18s). This suggests it over-retrieved documents, increasing noise, latency, and computational cost without improving performance.

Overall, I found that the ParentDocumentRetriever provides the best tradeoff between performance, cost, and latency for this dataset. While the Compression Retriever achieved perfect scores, the additional reranking step may not justify the extra cost in larger-scale systems. The Parent retriever delivers near-perfect results efficiently, making it the most practical choice for this use case.